In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

sns.set()

Este es un análisis del conjunto de datos Online Retail, proporcionado por la UCI, que contiene transacciones (incluidas cancelaciones) realizadas por un comercio minorista en línea (centrado principalmente en artículos de regalo) entre el 01/12/2009 y el 09/12/2011.



```
# Tiene formato de código
```

# Análisis exploratorio de datos

El conjunto de datos contiene 1.067.371 registros y tiene 8 columnas. Esta es una breve descripción de las columnas:

* Invoice: Número de factura. Nominal. Número entero de 6 dígitos asignado de forma exclusiva a cada transacción. Si el código empieza por la letra «C», indica una cancelación.
* StockCode: Código del producto (artículo). Nominal. Número entero de 5 dígitos asignado de forma exclusiva a cada producto distinto.
* Description: Nombre del producto (artículo). Nominal.
* Quantity: Cantidad de cada producto (artículo) por transacción. Numérica.
* InvoiceDate: Fecha y hora de la factura. Numérica. El día y la hora en que se generó una transacción.
* Price: Precio unitario. Numérico. Precio del producto por unidad en libras esterlinas (£).
* CustomerID: Número de cliente. Nominal. Número entero de 5 dígitos asignado de forma exclusiva a cada cliente.
* Country: Nombre del país. Nominal. Nombre del país donde reside el cliente.

In [7]:
df = pd.concat(pd.read_excel("data/online_retail_II.xlsx", sheet_name=None).values())

FileNotFoundError: [Errno 2] No such file or directory: 'data/online_retail_II.xlsx'

In [ ]:
df.columns

In [ ]:
df.shape

Voy a examinar las primeras 10 entradas del conjunto de datos y también 10 entradas elegidas al azar para comprobar si los datos están limpios.

In [ ]:
df.head(10)

In [ ]:
df.sample(10)

Observamos que faltan algunos valores en la columna **Customer ID**. Al llamar al método *info* del *dataframe* de Pandas vemos que la columna **Description** también contiene valores ausentes. De hecho, aproximadamente el 20 % de las filas tiene un valor ausente en la columna **Customer ID**, lo cual es mucho. Los tipos de datos de las columnas parecen válidos, salvo el de **Customer ID**, que es de tipo *float* cuando debería ser un entero de 5 dígitos.

In [ ]:
df.info()

Al llamar al método **describe** del *dataframe* de Pandas se muestran algunas estadísticas resumidas, como la media, la mediana y los cuartiles, para las columnas numéricas **Quantity** y **Price**. Esto muestra que ambas columnas tienen valores negativos, lo cual no tiene sentido: no se pueden comprar menos de 0 artículos ni a un precio negativo.

In [ ]:
df[["Quantity", "Price"]].describe()

Ahora exploramos los valores de las columnas del conjunto de datos y, en particular, los valores únicos de las columnas no numéricas. La empresa ha enviado pedidos a 43 países distintos, la mayoría de ellos en Europa.

In [8]:
df["Country"].nunique()

NameError: name 'df' is not defined

In [ ]:
df["Country"].unique()

92 % of the transactions are made with customers residing in the United Kingdom, other transactions are mostly made in neighbour countries such as Republic of Ireland, Germany or France.

In [ ]:
df["Country"].value_counts(normalize=True).head(10) * 100

In [ ]:
print(f"Transactions registered from {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")

In [ ]:
print(f"Number of transactions registered: {df['Invoice'].nunique()}")

El 92 % de las transacciones se realiza con clientes que residen en el Reino Unido. El resto de las transacciones procede principalmente de países vecinos, como la República de Irlanda, Alemania o Francia.

In [ ]:
ax = df.set_index('InvoiceDate').resample("M")["Invoice"].nunique().plot(marker=".",
                                                                         linestyle="--",
                                                                         markerfacecolor="black",
                                                                         markeredgecolor="black"
                                                                         )
ax.set_ylabel("Total Sales")
ax.set_xlabel("Month of the Year");

In [ ]:
print(f"Number of item descriptions: {df['Description'].nunique()}")

In [ ]:
print(f"Number of item stock code: {df['StockCode'].nunique()}")

In [9]:
print(f"Number of unique customers: {df['Customer ID'].nunique()}")

NameError: name 'df' is not defined

El número de transacciones se mantiene más o menos constante durante el año, pero aumenta de septiembre a diciembre, como cabría esperar, ya que en este periodo tienen lugar muchos acontecimientos importantes, como Halloween y Navidad.

Hay algunos valores ausentes en las columnas **Description** y **Customer ID**. Veamos las filas en las que faltan los valores de la columna **Description**.

In [ ]:
df["Description"].isnull().mean() * 100

#### Valores ausentes en la columna **Description**:

In [ ]:
df[df["Description"].isnull()].sample(10)

El 0,41 % de las filas de la primera hoja tiene valores ausentes en la columna **Description**. En las celdas siguientes se muestra un ejemplo de 10 de esas filas. Parece que esas filas tienen valores negativos en la columna **Quantity**, muchos valores 0,0 en la columna **Price** y valores ausentes en la columna **Customer ID**.

In [10]:
np.all(df[df["Description"].isnull()]["Customer ID"].isnull())

NameError: name 'df' is not defined

In [ ]:
(df[df["Description"].isnull()]["Quantity"] <= 0).mean() * 100

In [ ]:
all(df[df["Description"].isnull()]["Country"] == "United Kingdom")

In [ ]:
all(df[df["Description"].isnull()]["Price"] == 0.0)

El 61 % de las filas tiene un valor negativo en la columna **Quantity**. Además, la columna **Country** solo contiene *United Kingdom* y la columna **Price** está rellena de ceros. Estas filas podrían revelar patrones de datos erróneos, como cantidades negativas o productos con precio 0, que también podemos encontrar en otras filas. En este último caso, es posible que algunos artículos sean gratuitos; lo comprobaré en una exploración posterior.

#### Valores ausentes en la columna **Customer ID**:

Veamos ahora las filas cuyos únicos valores ausentes se encuentran en la columna **Customer ID**, que representan el 22 % de los datos.

In [ ]:
df["Customer ID"].isnull().mean() * 100

In [ ]:
df[df["Customer ID"].isnull()].sample(10)

In [11]:
df[df["Customer ID"].isnull()]["Description"].nunique()

NameError: name 'df' is not defined

In [ ]:
(df[df["Customer ID"].isnull()]["Quantity"] <= 0).mean() * 100

In [ ]:
(df[df["Customer ID"].isnull()]["Price"] <= 0.0).mean() * 100

Todas las filas contienen valores ausentes en la columna **Customer ID**, como se muestra a continuación.

In [ ]:
(df["Price"] <= 0).mean() * 100

In [ ]:
(df["Quantity"] <= 0).mean() * 100

In [ ]:
df[df["Quantity"] <= 0].sample(10)

Al observar la columna **Invoice**, vemos que en la mayoría de las filas (el 85 %) aparece una *C* delante del número y, según la descripción de los datos, esto significa que el pedido fue cancelado.

In [ ]:
df[df["Quantity"] <= 0]["Invoice"].astype("str").str.startswith("C").mean() * 100

In [ ]:
(df[df["Invoice"].astype("str").str.startswith("C")]["Quantity"] <= 0.).mean() * 100

Creemos una nueva columna llamada **TotalPrice**, que será igual a **Quantity** $\times$ **Price**. Las filas con un valor menor o igual que cero representan el 2,4 % del conjunto de datos e incluyen las filas con un precio negativo o una cantidad negativa.

In [ ]:
df["TotalPrice"] = df["Quantity"] * df["Price"]

In [ ]:
(df["TotalPrice"] <= 0).mean() * 100

In [ ]:
((df["Price"] <= 0) | (df["Quantity"] <= 0) ).mean() * 100

In [ ]:
df[df["TotalPrice"] <= 0]

Conservar todas las filas con **TotalPrice** $>$ 0 elimina todas las filas con valores ausentes en la columna **Description** y también todas las órdenes canceladas salvo una.

In [ ]:
all(~df[df["TotalPrice"] > 0]["Description"].isnull())

In [ ]:
df[(df["Invoice"].astype("str").str.startswith("C")) & (df["TotalPrice"] > 0)]

#### Exploración adicional con los datos limpios.

Creemos ahora una función que cargue los datos y filtre las filas con **TotalPrice** negativo y los pedidos cancelados. La función incluye un argumento para conservar las filas con valores ausentes en la columna **Customer ID**.

In [ ]:
def load_df(sheet_name="Year 2009-2010", keepna=True):

    df = pd.read_excel("data/online_retail_II.xlsx",
                       sheet_name=sheet_name,
                       parse_dates=["InvoiceDate"],
                       dtype={"Invoice": "str", "StockCode": "str"})

    if isinstance(df, dict):
        df = pd.concat(df.values())

    df["TotalPrice"] = df["Quantity"] * df["Price"]
    df = df.query("TotalPrice > 0")
    df = df[~df["Invoice"].str.startswith("C")]
    df = df[~df["StockCode"].str.contains("TEST")]

    if not keepna:
        df.dropna(inplace = True)

    df.drop_duplicates(inplace=True)

    assert all(df["InvoiceDate"] > datetime(2009,1,1))
    assert all(df["InvoiceDate"] < datetime(2012,1,1))

    return df

In [ ]:
df = load_df(None)

In [ ]:
df.info()

Now that we have prepared the dataset let's explore further. 22 % percent of the rows have missing values in the **Customer ID**. values which is a lot and quite unfortunate as we cannot track next orders from those customers.

Ahora que hemos preparado el conjunto de datos, podemos seguir explorándolo. El 22 % de las filas tiene valores ausentes en **Customer ID**, una proporción muy alta y bastante desafortunada, ya que no podemos hacer un seguimiento de los pedidos posteriores de esos clientes.

In [ ]:
df.dropna().groupby("Description")["Quantity"].sum().sort_values(ascending=False).head()

In [ ]:
df[df["Customer ID"].isnull()].groupby("Description")["Quantity"].sum().sort_values(ascending=False).head()

Hay un total de 100.000 pedidos en el conjunto de datos. A continuación se muestran los artículos más vendidos por los clientes con y sin **Customer ID**; observamos que no compran los mismos artículos.

In [ ]:
df.dropna().groupby("Description")["TotalPrice"].sum().sort_values(ascending=False).head()

In [12]:
df.loc[df["Description"] == "WORLD WAR 2 GLIDERS ASSTD DESIGNS"]["TotalPrice"].sum()

NameError: name 'df' is not defined

Sin embargo, el artículo que generó más ingresos para la empresa no es «WORLD WAR 2 GLIDERS ASSTD DESIGNS», sino «REGENCY CAKESTAND 3 TIER», con un total de 278.000 £. En comparación, «WORLD WAR 2 GLIDERS ASSTD DESIGNS» solo generó 25.000 £.

In [ ]:
df.sort_values("Price", ascending = False).head()

In [ ]:
df.dropna().sort_values("Price", ascending = False).head()

In [ ]:
bounds=(0, 26000)
bins=50
ax = df.dropna()["Price"].plot(kind="hist", range=bounds, bins=bins, density=True, label="Known customers")
_ = df[df["Customer ID"].isnull()]["Price"].plot(kind="hist", range=bounds, bins=bins, alpha=0.6, density=True,
                                                 label="Unknown customers")
ax.set_yscale("log")
ax.set_xscale("log")
ax.set_xlabel("Price in £")
_ = ax.legend()

In [ ]:
def ecdf(data):
    """ Compute ECDF """
    x = np.sort(data)
    n = x.size
    y = np.arange(1, n+1) / n
    return(x,y)

In [ ]:
price_known = df.dropna()["Price"].values
price_unknown = df[df["Customer ID"].isnull()]["Price"].values

f, ax = plt.subplots()
ax.scatter(*ecdf(price_known), label="Known customers")
ax.scatter(*ecdf(price_unknown), label="Unknown customers")
ax.set_xscale("log")
ax.set_ylabel("ECDF")
ax.set_xlabel("Price in £")
ax.legend()

Veamos cuáles son los artículos más caros. Como se muestra a continuación, los 5 artículos más caros fueron comprados por clientes desconocidos o no registrados, con precios de hasta 25.000 £. Si observamos únicamente a los clientes con un **Customer ID**, los 5 artículos más caros tienen precios inferiores a los de los clientes desconocidos. El gráfico siguiente muestra la distribución del precio por artículo para las filas con y sin valores ausentes en **Customer ID**. Las distribuciones están muy sesgadas, pero podemos ver que los clientes desconocidos compran artículos más caros que los clientes identificados.

In [13]:
bounds=(0, 40000)
bins=50
ax = df.dropna()["Quantity"].plot(kind="hist", range=bounds, bins=bins, density=True, label="Known customers", alpha=0.7)
_ = df[df["Customer ID"].isnull()]["Quantity"].plot(kind="hist", range=bounds, bins=bins, density=True,
                                                    label="Unknown customers", alpha=0.8)
ax.set_yscale("log")
ax.set_xscale("log")
ax.set_xlabel("Quantity")
_ = ax.legend()

NameError: name 'df' is not defined

In [ ]:
quant_known = df.dropna()["Quantity"].values
quant_unknown = df[df["Customer ID"].isnull()]["Quantity"].values

f, ax = plt.subplots()
ax.scatter(*ecdf(quant_known), label="Known customers")
ax.scatter(*ecdf(quant_unknown), label="Unknown customers")
ax.set_xscale("log")
ax.set_ylabel("ECDF")
ax.set_xlabel("Quantity")
ax.legend()

El gráfico siguiente muestra que los clientes cuyo **Customer ID** es desconocido también piden menos artículos que los clientes registrados.

In [ ]:
df_inv = df.dropna().groupby("Invoice").agg({"TotalPrice": "sum", "Quantity":"sum"})
df_inv_na = df[df["Customer ID"].isnull()].groupby("Invoice").agg({"TotalPrice": "sum", "Quantity":"sum"})

bounds=(0, 27000)
bins=40
ax = df_inv["TotalPrice"].plot(kind="hist", range=bounds, bins=bins, density=True, label="Known customers")
_ = df_inv_na["TotalPrice"].plot(kind="hist", range=bounds, bins=bins, alpha=0.6, density=True, label="Unknown customers")
ax.set_yscale("log")
#ax.set_xscale("log")
ax.set_xlabel("Price in £")
_ = ax.legend()

In [ ]:
totprice_known = df.dropna().groupby("Invoice")["TotalPrice"].sum().values
totprice_unknown = df[df["Customer ID"].isnull()].groupby("Invoice")["TotalPrice"].sum().values

f, ax = plt.subplots()
ax.scatter(*ecdf(totprice_known), label="Known customers")
ax.scatter(*ecdf(totprice_unknown), label="Unknown customers")
ax.set_xscale("log")
ax.set_ylabel("ECDF")
ax.set_xlabel("Price in £")
ax.legend()

Veamos el precio total y la cantidad total de artículos pedidos por factura. Los clientes sin **Customer ID** pagan facturas más elevadas que los clientes registrados.

In [ ]:
ax = df_inv.plot(kind="scatter", x="Quantity", y="TotalPrice", alpha=0.8, label="known customers")
df_inv_na.plot(kind="scatter", x="Quantity", y="TotalPrice", ax=ax, color="crimson", alpha=0.8, label="Unknown customers")
ax.set_ylim(-100, 30000)
ax.set_xlim(-100, 20000)
ax.set_ylabel("Total price payed per transaction")
ax.set_xlabel("Total number of items ordered per transaction")
_ = ax.legend()

He calculado las «trayectorias de actividad» de cada cliente (la probabilidad de que siga activo); a continuación se muestra un ejemplo de esas trayectorias. Las líneas verticales rojas representan la fecha de cada compra.

# Segmentación

Una forma de crear segmentos de clientes basados en sus patrones de compra es realizar un análisis RFM (recencia, frecuencia y valor monetario). Agrupa a los clientes según sus transacciones de compra anteriores. Estas son las definiciones de cada término del análisis RFM:

* Recencia: Tiempo transcurrido desde la última transacción del cliente (tomando como fecha de referencia el 10 de diciembre de 2011).

* Frecuencia: Número total de transacciones.

* Valor monetario: Gasto total del cliente.

Crearemos un nuevo *dataframe* con estas métricas calculadas para cada cliente. Por tanto, debemos eliminar las filas con valores ausentes en la columna **Customer ID**.

In [ ]:
from datetime import timedelta
snapshot_date = max(df.InvoiceDate) +timedelta(days=1)
#snapshot_date = datetime(2012, 1, 1)

In [ ]:
rfm = df.dropna().groupby("Customer ID").agg({"Invoice": lambda x: x.nunique(),
                                              "InvoiceDate": lambda date: (snapshot_date - date.max()).days,
                                              "TotalPrice": "sum"})

rfm.rename(columns= {'InvoiceDate': 'recency',
                     'Invoice': 'frequency',
                      'TotalPrice': 'monetary'}, inplace= True)

rfm.head(5)

In [ ]:
rfm.describe()

Representemos las distribuciones de las tres cantidades.

In [ ]:
f, ax = plt.subplots(1, 3, figsize=(20, 6))
for a in ax:
    a.set_ylabel("# of occurences")

sns.distplot(rfm.query("frequency < 100")["frequency"], kde=False, ax=ax[0])

sns.distplot(rfm["recency"], kde=False, ax=ax[1])

sns.distplot(rfm.query("monetary < 20000")["monetary"], kde=False, ax=ax[2]);

La siguiente tarea consiste en encontrar agrupaciones en los valores RFM; para ello utilizaremos el algoritmo KMeans de scikit-learn.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from scipy import stats

Supuestos del algoritmo KMeans:
* Distribución simétrica de las variables (sin sesgo).
* Variables con los mismos valores medios.
* Variables con la misma varianza.

Por tanto, debemos corregir el sesgo de las variables (mediante una transformación logarítmica) y estandarizarlas después.

In [ ]:
def plot_dist_transformed(transformation=lambda x: x):

    f, ax = plt.subplots(1, 3, figsize=(20, 6))
    for i, a in enumerate(ax):
        a.set_ylabel("# of occurences")

        col = rfm.columns[i]
        sns.distplot(transformation(rfm[col]), kde=False, ax=a)
        a.set_xlabel(col)

plot_dist_transformed(lambda x: stats.boxcox(x)[0])

In [ ]:
rfm_unskewed = np.array([stats.boxcox(rfm[["frequency", "recency", "monetary"]].values[:,i])[0] for i in range(3)]).T

In [ ]:
scaler = StandardScaler()
scaler.fit(rfm_unskewed)
rfm_scaled = pd.DataFrame(scaler.transform(rfm_unskewed), columns=["frequency", "recency", "monetary"])

In [ ]:
def plot_dist_scaled(rfms=rfm_scaled):

    f, ax = plt.subplots(1, 3, figsize=(20, 6))
    for i, a in enumerate(ax):
        a.set_ylabel("# of occurences")

        col = rfms.columns[i]
        sns.distplot(rfms[col], kde=False, ax=a)
        a.set_xlabel(col)

plot_dist_scaled()

Ahora necesitamos encontrar el número óptimo de agrupaciones. Para ello calcularemos distintas métricas:

* puntuación de silueta (cuanto mayor, mejor);
* puntuación de Calinski-Harabasz (cuanto mayor, mejor);
* puntuación de Davies-Bouldin (cuanto menor, mejor);
* método del codo aplicado a la métrica de inercia (el mejor número de agrupaciones se encuentra en el codo o cambio de pendiente).

In [ ]:
inertias = []
silhouettes = []
calinski = []
davies = []
n_clusters = np.arange(2, 10, 1)

for n in n_clusters:
    kmeans = KMeans(n_clusters=n)
    kmeans.fit(rfm_scaled)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(rfm_scaled, kmeans.predict(rfm_scaled)))
    calinski.append(calinski_harabasz_score(rfm_scaled, kmeans.predict(rfm_scaled)))
    davies.append(davies_bouldin_score(rfm_scaled, kmeans.predict(rfm_scaled)))

In [ ]:
f, ax = plt.subplots(2, 2, figsize=(20, 10))
for a in ax.flatten():
    a.set_xlabel("Number of clusters")

ax[0, 0].plot(n_clusters, silhouettes, ".--")
ax[0, 0].set_ylabel("Silhouette score")

ax[0, 1].plot(n_clusters, calinski, ".--")
ax[0, 1].set_ylabel("Calinski and Harabasz score")

ax[1, 0].plot(n_clusters, davies, ".--")
ax[1, 0].set_ylabel("Davies-Bouldin score")

ax[1, 1].plot(n_clusters, inertias, ".--")
ax[1, 1].set_ylabel("Inertia score")

El gráfico sugiere que el mejor número de agrupaciones es 4. KMeans calculará y asignará una agrupación a cada cliente.

In [ ]:
nclusters=4
kmeans = KMeans(n_clusters=nclusters, random_state=13)
kmeans.fit(rfm_scaled)

rfm["cluster"] = kmeans.predict(rfm_scaled)

In [ ]:
rfm.head()

A continuación se muestra el número de clientes de cada agrupación.

In [ ]:
rfm["cluster"].value_counts()

In [14]:
rfm.groupby("cluster").mean()

NameError: name 'rfm' is not defined

Hemos calculado el valor medio de las métricas RFM para cada agrupación de clientes:

La agrupación n.º 0 parece ser la mejor: contiene a la mayoría de las personas, es el grupo que más dinero gasta y que ha realizado más transacciones y, en promedio, realizó su última transacción hace 34 días. Son los **clientes habituales**; se pueden tomar decisiones de marketing para ellos, como ofrecerles envío gratuito.

La agrupación n.º 2 es la siguiente mejor, porque sus clientes gastan mucho dinero. Sin embargo, hace tiempo que los clientes de este grupo no compran. Podría ser buena idea recordarles la existencia de la empresa y proponerles nuevos productos u ofertas para trasladarlos a otro grupo.

La agrupación n.º 3 es la siguiente mejor después de la n.º 2. Sus clientes gastaron menos, pidieron menos artículos y realizaron su última transacción hace varios meses. Podrían clasificarse como **clientes ocasionales**, que compran de vez en cuando.

La agrupación n.º 1 gastó poco y hace mucho tiempo que no compra; en promedio, probablemente ya han dejado de utilizar el servicio y han abandonado la empresa.

Representemos las distribuciones normalizadas de las métricas RFM para cada agrupación.

In [ ]:
f, ax = plt.subplots(1, 3, figsize=(20, 6))
for a in ax:
    a.set_ylabel("density")

for c in range(nclusters):
    rfm_c =  rfm.query(f"cluster == {c}")
    label = f"Cluster {c}"

    sns.distplot(rfm_c.query("frequency < 30")["frequency"], kde=True, ax=ax[0],
                 label=label, hist=False, norm_hist=True)

    sns.distplot(rfm_c["recency"], kde=True, ax=ax[1], label=label, hist=False,
                 norm_hist=True)

    sns.distplot(rfm_c.query("monetary < 10000")["monetary"], kde=True, ax=ax[2],
                 label=label, hist=False, norm_hist=True)

for a in ax:
    a.legend();

In [ ]:
rfm

In [ ]:
rfm

In [ ]:
def plot_dist_transformed_cluster(transformation=lambda x: x, columns=["frequency", "recency", "monetary"]):

    f, ax = plt.subplots(1, 3, figsize=(20, 6))

    bins = 30

    ranges = {}

    rfm_ = pd.DataFrame(scaler.transform(transformation(rfm[columns])), columns=columns)
    rfm_["cluster"] = rfm["cluster"].values


    for i, a in enumerate(ax):
        a.set_ylabel("# of occurences")
        col = rfm_.columns[i]
        x_min, y_min = min(rfm_[col]), max(rfm_[col])
        ranges[col] = (x_min, y_min)
        a.set_xlim(x_min, y_min)
        a.set_xlabel(col)

    for c in range(nclusters):
        rfm_c =  rfm_.query(f"cluster == {c}")
        label = f"Cluster {c}"

        for i, a in enumerate(ax):
            col = rfm_c.columns[i]
            toplot = rfm_c[col]
            sns.distplot(toplot, kde=True, ax=a, label=label, hist=False)#, bins=bins, hist_kws=dict(range=ranges[col]))
            a.legend()

plot_dist_transformed_cluster(lambda x: np.array([stats.boxcox(x.values[:,i])[0] for i in range(3)]).T)

# Predicción de abandono

Para predecir si un cliente está activo o inactivo (si sigue activo o ha abandonado) en un negocio no contractual, podemos utilizar el análisis RFM. Los modelos BG/NBD, descritos [aquí](http://brucehardie.com/papers/018/fader_et_al_mksc_05.pdf), pueden calcular el «valor del tiempo de vida del cliente». Es la primera vez que trabajo con esto, así que todavía no conozco todos los detalles, pero el modelo y otras utilidades están implementados en el paquete [lifetimes](https://lifetimes.readthedocs.io/en/latest/index.html).

Construiremos un conjunto de datos RFM ligeramente distinto utilizando las utilidades de *lifetimes*, donde:

* *frequency* representa el número de compras repetidas que ha realizado el cliente. Esto significa que es uno menos que el número total de compras. En realidad, esto no es del todo correcto: es el número de periodos de tiempo en los que el cliente realizó una compra. Por ejemplo, si la unidad son días, es el número de días en los que realizó alguna compra.
* *T* representa la antigüedad del cliente en las unidades de tiempo elegidas (semanas en el conjunto de datos anterior). Es igual a la duración entre la primera compra del cliente y el final del periodo estudiado.
* *recency* representa la antigüedad del cliente cuando realizó sus compras más recientes. Es igual a la duración entre la primera compra del cliente y su última compra. Por tanto, si solo ha realizado una compra, la recencia es 0.
* *monetary_value* representa el valor medio de las compras de un cliente. Es igual a la suma de todas las compras del cliente dividida por el número total de compras. Ten en cuenta que el denominador es diferente de la frecuencia descrita anteriormente.

In [ ]:
from lifetimes.utils import summary_data_from_transaction_data
rfml = summary_data_from_transaction_data(df.dropna(), "Customer ID", "InvoiceDate", "TotalPrice")

In [ ]:
rfml.head()

In [ ]:
rfml.describe()

Ahora ajustaremos el modelo y obtendremos un resumen de sus parámetros ajustados y de la incertidumbre asociada.

In [ ]:
from lifetimes import BetaGeoFitter

bgf = BetaGeoFitter(penalizer_coef=0.0)
bgf.fit(rfml["frequency"], rfml["recency"], rfml["T"])

bgf.summary

A partir del modelo ajustado podemos hacer muchas cosas interesantes, como calcular el número esperado de transacciones que realizará un cliente hipotético en el siguiente periodo (1 día), dadas su recencia (antigüedad en la última compra) y su frecuencia (el número de transacciones repetidas que ha realizado).

In [ ]:
from lifetimes.plotting import plot_frequency_recency_matrix
f = plt.figure(figsize=(10, 5))
plot_frequency_recency_matrix(bgf, cmap="viridis");

Podemos observar que, si un cliente ha comprado aproximadamente 250 veces en el comercio y su última compra fue cuando tenía algo más de 700 semanas (dado que el cliente tiene unos 700 años de antigüedad), entonces es uno de los mejores clientes (abajo a la derecha). Los clientes menos activos son los de la esquina superior derecha: compraron mucho en poco tiempo y no los hemos vuelto a ver durante semanas.

También podemos representar la probabilidad de que sigan activos.

In [ ]:
from lifetimes.plotting import plot_probability_alive_matrix
f = plt.figure(figsize=(10, 5))
plot_probability_alive_matrix(bgf, cmap="viridis");

### ¿Han abandonado algunos usuarios durante la segunda mitad de 2011?

Para determinarlo, utilizaré los historiales de probabilidad de que los clientes sigan activos en función del tiempo, calculados a partir del modelo BG/NBD.

In [ ]:
from lifetimes.utils import calculate_alive_path
from tqdm import tqdm

customer_ids = df.dropna()["Customer ID"].unique()

alive_paths = {}
for i, cid in enumerate(tqdm(customer_ids)):
    days_since_birth = int(rfml.loc[cid]["T"])
    sp_trans = df[df["Customer ID"] == cid]
    alive_paths[cid] = calculate_alive_path(bgf, sp_trans, "InvoiceDate", days_since_birth)

He calculado las «trayectorias de actividad» de cada cliente (la probabilidad de que siga activo); a continuación se muestra un ejemplo de esas trayectorias. Las líneas verticales rojas representan la fecha de cada compra.

In [ ]:
from lifetimes.plotting import plot_history_alive

def plot_path(cid, ax=None, july_line=False):
    days_since_birth = int(rfml.loc[cid]["T"])
    sp_trans = df[df["Customer ID"] == cid]
    plot_history_alive(bgf, days_since_birth, sp_trans, "InvoiceDate", ax=ax)
    if july_line:
        plt.vlines(datetime(2011, 7, 1), 0, 1, color="forestgreen")

In [ ]:
plot_path(13085)

Recopilaré el **Customer ID** de los clientes que probablemente hayan abandonado el servicio durante la segunda mitad de 2011. Para ello, debo descartar a los clientes que abandonaron antes de ese periodo.

In [ ]:
inactive_after_july = [
    cid for cid, path in alive_paths.items()
    if path[-1] < 0.5
    and df.loc[df["Customer ID"] == cid, "InvoiceDate"].max() >= datetime(2011, 7, 1)
]

In [ ]:
print(f"Number of users that probably churned in the second half of 2011: {len(inactive_after_july)}")

Representemos las «trayectorias de actividad» de esos usuarios para comprobar que probablemente abandonaron el servicio durante este periodo.

In [15]:
f = plt.figure(figsize=(10, 5))
plot_path(inactive_after_july[0], july_line=True)

NameError: name 'plot_path' is not defined

<Figure size 1000x500 with 0 Axes>

In [ ]:
f = plt.figure(figsize=(10, 5))
plot_path(inactive_after_july[5], july_line=True)

In [ ]:
f = plt.figure(figsize=(10, 5))
plot_path(inactive_after_july[10], july_line=True)

### ¿Hay usuarios con un alto riesgo de abandono a finales de 2011?

Para determinarlo, debemos reunir a los clientes que siguieron activos hasta finales de 2011 y que tienen un número esperado bajo de transacciones futuras. En este caso, el futuro son 22 días, ya que la última fecha del conjunto de datos es el 9 de diciembre de 2011.

In [ ]:
alive_after_july = [cid for cid, path in alive_paths.items() if path[-1] >= 0.5]

Calculamos el número esperado de compras en los próximos 22 días y clasificamos a los clientes.

In [ ]:
t = 22
rfml["predicted_purchases"] = bgf.conditional_expected_number_of_purchases_up_to_time(t, rfml["frequency"], rfml["recency"], rfml["T"])
rfml.sort_values(by="predicted_purchases", ascending=False).head(5)

A continuación se muestra la distribución del número previsto de transacciones en los próximos 22 días.

In [ ]:
ax = rfml.loc[alive_after_july, "predicted_purchases"].plot(kind="hist", bins=50)
ax.set_xlabel("predicted number of transactions (in 22 days)")

Representemos las distribuciones normalizadas del número previsto de transacciones en los próximos 22 días para los clientes que probablemente seguían activos durante la segunda mitad de 2011, agrupadas por cada grupo de segmentación definido anteriormente.

In [ ]:
rfml.loc[alive_after_july]["cluster"].value_counts()

In [ ]:
f = plt.figure(figsize=(10, 5))

for c in range(5):
    rfm_c =  rfml.loc[alive_after_july].query(f"cluster == {c}")
    label = f"Cluster {c}"

    sns.distplot(rfm_c["predicted_purchases"], kde=True,
                 label=label, hist=False, norm_hist=True)
plt.xlim(0, 0.7)
plt.xlabel("predicted number of transactions (in 22 days)")

Como vemos, los miembros de la agrupación 3 tienen un mayor riesgo de abandono durante los próximos 22 días. También podríamos seleccionar una fracción de los clientes con el menor número previsto de transacciones y actuar para conservarlos.

In [16]:
rfml.loc[alive_after_july]["cluster"].value_counts()

NameError: name 'rfml' is not defined

In [ ]:
rfml.loc[alive_after_july].groupby("cluster").mean()

In [ ]:
rfm.groupby("cluster").mean()

### Valor del tiempo de vida del cliente

Calculemos el CLV de cada cliente y comparémoslo entre las distintas agrupaciones.

In [ ]:
rfml[['monetary_value', 'frequency']].corr()

In [ ]:
from lifetimes import GammaGammaFitter

rfml_m = rfml[rfml.monetary_value > 0]


ggf = GammaGammaFitter(penalizer_coef = 0)
ggf.fit(rfml_m['frequency'],
        rfml_m['monetary_value'])
print(ggf)

In [ ]:
print(ggf.conditional_expected_average_profit(
        rfml_m['frequency'],
        rfml_m['monetary_value']
    ).head(10))

In [ ]:
print("Expected conditional average profit: %s, Average profit: %s" % (
    ggf.conditional_expected_average_profit(
        rfml_m['frequency'],
        rfml_m['monetary_value']
    ).mean(),
    rfml_m[rfml_m['frequency']>0]['monetary_value'].mean()
))

In [ ]:
bgf_clv = BetaGeoFitter(penalizer_coef=0.0)
bgf_clv.fit(rfml_m['frequency'], rfml_m['recency'], rfml_m['T'])

print(ggf.customer_lifetime_value(
    bgf_clv, #the model to use to predict the number of future transactions
    rfml_m['frequency'],
    rfml_m['recency'],
    rfml_m['T'],
    rfml_m['monetary_value'],
    time=1, # months
    discount_rate=0.01 # monthly discount rate ~ 12.7% annually
).head(10))

In [ ]:
predicted_sum  = ggf.customer_lifetime_value(
    bgf_clv, #the model to use to predict the number of future transactions
    rfml_m['frequency'],
    rfml_m['recency'],
    rfml_m['T'],
    rfml_m['monetary_value'],
    time=1, # months
    discount_rate=0.01 # monthly discount rate ~ 12.7% annually
)

predicted_sum = pd.DataFrame(predicted_sum)

In [ ]:
ids = predicted_sum.index
predicted_sum.loc[ids, "cluster"] = rfm.loc[ids, "cluster"]

In [ ]:
f, ax = plt.subplots(figsize=(10, 5))

for c in range(5):
    p_c =  predicted_sum.query(f"cluster == {c}")
    label = f"Cluster {c}"

    sns.distplot(p_c["clv"], kde=True,
                 label=label, hist=False, ax=ax)
ax.set_xlim(10, 10000)
ax.set_xscale("log")
ax.set_xlabel("predicted CLV (in 1 month)")

In [ ]:
predicted_sum.groupby("cluster").agg({"clv": ["mean", "std", "median", "sum"]})

In [ ]:
predicted_sum["cluster"].value_counts()